In [1]:
import tensorflow as tf
import numpy as np
from PIL import Image


In [2]:
# Load pretrained model
model = tf.keras.applications.MobileNetV2(weights="imagenet")
print("Model ready")


14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Model ready


In [3]:
# Class labels are returned by ImageNet decoder
def predict_image(path):
    image = Image.open(path).convert("RGB").resize((224, 224))
    x = np.array(image, dtype=np.float32)

    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = np.expand_dims(x, axis=0)

    prediction = model.predict(x, verbose=0)

    results = tf.keras.applications.mobilenet_v2.decode_predictions(
        prediction, top=5
    )[0]

    return [
        {"label": label, "confidence": float(score)}
        for _, label, score in results
    ]


In [5]:
# Leaky API: exposes labels and confidence scores
def leaky_api(path):
    return predict_image(path)

print("Leaky API:", leaky_api("dog.jpg"))

35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
Leaky API: [{'label': 'stingray', 'confidence': 0.2026062309741974}, {'label': 'tree_frog', 'confidence': 0.04635056480765343}, {'label': 'tiger_shark', 'confidence': 0.02375795878469944}, {'label': 'spotlight', 'confidence': 0.017944950610399246}, {'label': 'vase', 'confidence': 0.017217934131622314}]


In [9]:
# Leaky API
def leaky_api(path):
    return predict_image(path)

# Protected API
def protected_api(path):
    result = predict_image(path)

    if isinstance(result, dict):
        return {"error": "Invalid input"}

    return {"label": result[0]["label"]}

# Test valid image
print("Leaky API:", leaky_api("dog.jpg"))
print("Protected API:", protected_api("dog.jpg"))

# Test invalid image
print("Leaky API error:", leaky_api("fake.jpg"))
print("Protected API error:", protected_api("fake.jpg"))

Leaky API: [{'label': 'stingray', 'confidence': 0.2026062309741974}, {'label': 'tree_frog', 'confidence': 0.04635056480765343}, {'label': 'tiger_shark', 'confidence': 0.02375795878469944}, {'label': 'spotlight', 'confidence': 0.017944950610399246}, {'label': 'vase', 'confidence': 0.017217934131622314}]
Protected API: {'label': 'stingray'}


UnidentifiedImageError: cannot identify image file 'fake.jpg'